# 5. Modeling Phase: K-Means Customer Segmentation
## Unsupervised Clustering (CRISP-DM)
With our dataset reduced to 7 orthogonal principal components via PCA, we now apply **K-Means clustering** to discover natural customer segments. The workflow:
1. Find the optimal number of clusters using the **Elbow Method** and **Silhouette Analysis**.
2. Fit the final model and assign cluster labels.
3. Map labels back to original business features for interpretable profiling.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded.")


### 5.1 Load PCA-Reduced Feature Matrix


In [ ]:
data_dir = 'data'

pca_df = pd.read_csv(os.path.join(data_dir, 'customer_pca_features.csv'))
if 'CustomerID' in pca_df.columns:
    pca_df = pca_df.set_index('CustomerID')

X = pca_df.values
print(f"PCA matrix loaded: {X.shape[0]} customers × {X.shape[1]} components")


### 5.2 Elbow Method (WCSS / Inertia)
We run K-Means for k = 2 to 10 and record the **Within-Cluster Sum of Squares (WCSS)**. The "elbow" — where WCSS stops falling sharply — indicates the point of diminishing returns.


In [ ]:
K_range = range(2, 11)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))

# Plot Elbow
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method')

axes[1].plot(K_range, sil_scores, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by k')

plt.tight_layout()
plt.show()

# Print scores for easy comparison
for k, inertia, sil in zip(K_range, inertias, sil_scores):
    print(f"  k={k}  |  Inertia={inertia:.1f}  |  Silhouette={sil:.4f}")


### 5.3 Optimal k Selection
Examine the plots above to find the best trade-off between:
- **Elbow point** (where the inertia curve flattens)
- **Highest Silhouette Score** (higher = better-separated clusters)

Set `optimal_k` below based on your judgement of these two diagnostics.


In [ ]:
# Choose optimal k based on the highest silhouette score
optimal_k = K_range[np.argmax(sil_scores)]
print(f"Automatically selected optimal k = {optimal_k} (highest silhouette score: {max(sil_scores):.4f})")
print("You may override this value if the Elbow plot suggests a different k.")


### 5.4 Final K-Means Model


In [ ]:
kmeans_final = KMeans(n_clusters=optimal_k, n_init=20, random_state=42)
cluster_labels = kmeans_final.fit_predict(X)

pca_df['Cluster'] = cluster_labels
print(f"Final K-Means fitted with k={optimal_k}")
print(f"Cluster distribution:\n{pd.Series(cluster_labels).value_counts().sort_index()}")


### 5.5 Cluster Visualization (PC1 vs PC2)
Projecting clusters onto the first two principal components provides an intuitive 2-D view of how well the segments are separated.


In [ ]:
plt.figure(figsize=(12, 8))

scatter = plt.scatter(pca_df.iloc[:, 0], pca_df.iloc[:, 1],
                      c=cluster_labels, cmap='tab10', alpha=0.75, edgecolors='k', linewidth=0.5, s=80)

# Plot centroids
centroids = kmeans_final.cluster_centers_
plt.scatter(centroids[:, 0], centroids[:, 1],
            c='red', marker='X', s=250, edgecolors='black', linewidths=1.5, label='Centroids')

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title(f'K-Means Clustering (k={optimal_k}) — PC1 vs PC2')
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.tight_layout()
plt.show()


### 5.6 Cluster Business Profiling
To make the clusters **interpretable**, we map labels back to the original un-scaled customer features (Frequency, Expenditure, AOV, Region, Category). This lets us name each segment in business terms.


In [ ]:
# Reload original Gold customer profiles (before scaling)
customers = pd.read_csv(os.path.join(data_dir, 'Customers.csv'))
products = pd.read_csv(os.path.join(data_dir, 'Products.csv'))
transactions = pd.read_csv(os.path.join(data_dir, 'Transactions.csv'))

# Merge
master = transactions.merge(customers, on='CustomerID', how='left')
master = master.merge(products, on='ProductID', how='left')

# Drop temporal columns
for col in ['TransactionDate', 'SignupDate']:
    if col in master.columns:
        master.drop(columns=[col], inplace=True)

def get_mode(s):
    return s.mode()[0] if not s.empty else None

# Rebuild the Gold FM table
gold = master.groupby('CustomerID').agg(
    Frequency=('TransactionID', 'count'),
    Total_Expenditure=('TotalValue', 'sum'),
    Average_Order_Value=('TotalValue', 'mean'),
    Total_Items_Bought=('Quantity', 'sum'),
    Region=('Region', get_mode),
    Preferred_Category=('Category', get_mode)
).reset_index()

# Attach cluster labels
gold['Cluster'] = gold['CustomerID'].map(pca_df['Cluster'])

print("Cluster labels mapped to business profiles.")
gold.head()


### 5.7 Segment Summary Statistics


In [ ]:
# Numerical summary per cluster
cluster_summary = gold.groupby('Cluster').agg(
    Count=('CustomerID', 'count'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Total_Expenditure=('Total_Expenditure', 'mean'),
    Avg_AOV=('Average_Order_Value', 'mean'),
    Avg_Items=('Total_Items_Bought', 'mean')
).round(2)

print("\n=== Cluster Numerical Profiles ===")
print(cluster_summary.to_string())

# Categorical mode per cluster
print("\n=== Dominant Region per Cluster ===")
print(gold.groupby('Cluster')['Region'].agg(lambda x: x.mode()[0]))

print("\n=== Dominant Category per Cluster ===")
print(gold.groupby('Cluster')['Preferred_Category'].agg(lambda x: x.mode()[0]))


### 5.8 Cluster Behavioral Distributions


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.boxplot(x='Cluster', y='Frequency', data=gold, palette='tab10', ax=axes[0, 0])
axes[0, 0].set_title('Frequency by Cluster')

sns.boxplot(x='Cluster', y='Total_Expenditure', data=gold, palette='tab10', ax=axes[0, 1])
axes[0, 1].set_title('Total Expenditure by Cluster')

sns.boxplot(x='Cluster', y='Average_Order_Value', data=gold, palette='tab10', ax=axes[1, 0])
axes[1, 0].set_title('Average Order Value by Cluster')

sns.boxplot(x='Cluster', y='Total_Items_Bought', data=gold, palette='tab10', ax=axes[1, 1])
axes[1, 1].set_title('Total Items Bought by Cluster')

plt.tight_layout()
plt.show()


### 5.9 Export Labeled Dataset
Saving the cluster-labeled customer profiles for the **Decision Tree** interpretation phase.


In [ ]:
export_path = os.path.join(data_dir, 'customer_clusters.csv')
gold.to_csv(export_path, index=False)
print(f"Labeled dataset exported to: {export_path}")
